# Delete generated Zarr movies from the NAS

Use this notebook **after you have finished training, validation, and analysis** that need these movies. Run the cells from top to bottom.

It reads the exact paths in `scripts/processed_zarr_files.txt`, written by `scripts/convert_missing_files.py`. It does not search the NAS for other Zarr stores. The converter replaces this list when it starts a new run, so check that you have the list for the run you want to clean up.

A Zarr “file” is a folder containing many files. Deleting it permanently removes that whole folder, without using the Trash. Raw movies, TIFF masks, bounding boxes, and processing settings alongside it are kept.

**The notebook starts in dry-run mode: it previews what would be deleted.** It cannot tell whether your work is finished or whether you have a backup. Make sure you no longer need these generated movies before enabling deletion.


## 1. Choose the file list and NAS location

Open this notebook from the project folder, using a Python 3 kernel. Only Python's standard library is needed.

The defaults match this project. If your NAS has a different mount location, update `NAS_ROOT` and the paths in your list to match. The paths must be absolute. Keep the root as specific as possible.


In [4]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import csv
import shutil

PROJECT_ROOT = Path('../')
FILE_LIST = PROJECT_ROOT / "scripts" / "processed_zarr_files.txt"
NAS_ROOT = Path("/Volumes/Data/Users/Francesca/Data")
ZARR_NAME = "1-jumpCorrected.ome.zarr"
LOG_DIR = PROJECT_ROOT / "cleanup_logs"

print(f"File list: {FILE_LIST}")
print(f"Allowed NAS folder: {NAS_ROOT}")

if not FILE_LIST.is_file():
    raise FileNotFoundError(
        "File list not found. Open the notebook from the project folder, "
        "or set PROJECT_ROOT to that folder's absolute path."
    )


File list: ../scripts/processed_zarr_files.txt
Allowed NAS folder: /Volumes/Data/Users/Francesca/Data


## 2. Check the paths

This helper limits deletion to `processedMovies/1-jumpCorrected.ome.zarr` inside your chosen NAS folder. It refuses symbolic links in the path, unexpected names, and folders without Zarr metadata.

Missing stores are skipped, which makes it possible to run this notebook again after cleanup. If a path is blocked, read the reason before proceeding.


In [5]:
def check_target(path, root):
    """Return a preview status and a plain-language explanation."""
    if not path.is_absolute() or ".." in path.parts:
        return "blocked", "Use an absolute path without '..'."

    try:
        path.relative_to(root)
    except ValueError:
        return "blocked", "Outside the allowed NAS folder."

    if path.name != ZARR_NAME or path.parent.name != "processedMovies":
        return "blocked", "Unexpected Zarr name or parent folder."

    try:
        # Check the store itself and every parent, including the NAS root.
        if any(part.is_symlink() for part in (path, *path.parents)):
            return "blocked", "A symbolic link occurs in this path."
        if not path.exists():
            return "missing", "Already removed, or path not found."
        if not path.is_dir():
            return "blocked", "Expected a Zarr directory."
        if not any((path / name).is_file()
                   for name in ("zarr.json", ".zgroup", ".zarray")):
            return "blocked", "No Zarr metadata found in this directory."
    except OSError as error:
        return "blocked", f"Cannot inspect this path: {error}"

    return "ready", "Listed Zarr directory inside the allowed NAS folder."


## 3. Preview the cleanup

Mount the NAS first. This cell reads the list and checks the stores; it does not delete anything. Repeated paths are counted once.

Review **every path marked READY** below. Those are the only stores the deletion cell can remove. This preview does not read movie pixels or calculate folder sizes, so it avoids scanning every chunk on the NAS.

If you change the settings or file list, run this preview again.


In [6]:
# Clear any previous preview before doing new checks.
preview_paths = ()
preview_settings = None
preview_list_text = None

if not NAS_ROOT.is_absolute() or NAS_ROOT.resolve() != NAS_ROOT:
    raise ValueError("NAS_ROOT must be an absolute path without symbolic links or '..'.")
if not NAS_ROOT.is_dir():
    raise FileNotFoundError(f"NAS folder unavailable. Mount the NAS and check: {NAS_ROOT}")

list_text = FILE_LIST.read_text()
paths = tuple(dict.fromkeys(
    Path(line.strip()) for line in list_text.splitlines() if line.strip()
))
if not paths:
    raise ValueError("The file list is empty. There is nothing to clean up.")

preview_rows = [(path, *check_target(path, NAS_ROOT)) for path in paths]
counts = Counter(status for _, status, _ in preview_rows)

for index, (path, status, reason) in enumerate(preview_rows, start=1):
    print(f"{index:03d}  {status.upper():7}  {path}")
    if status != "ready":
        print(f"     {reason}")

preview_paths = tuple(path for path, status, _ in preview_rows if status == "ready")
preview_settings = (FILE_LIST, NAS_ROOT, ZARR_NAME)
preview_list_text = list_text
print(f"\nUnique paths: {len(paths)}")
print(f"Ready: {counts['ready']} | Missing: {counts['missing']} | Blocked: {counts['blocked']}")
print("No files have been deleted.")


001  READY    /Volumes/Data/Users/Francesca/Data/2024/2024_07_05_Pax2-GCaMP6f_P8_in vivo_injectable anaesthetic/1/processedMovies/1-jumpCorrected.ome.zarr
002  READY    /Volumes/Data/Users/Francesca/Data/2024/2024_07_05_Pax2-GCaMP6f_P8_in vivo_injectable anaesthetic/1_001/processedMovies/1-jumpCorrected.ome.zarr
003  READY    /Volumes/Data/Users/Francesca/Data/2024/2024_07_05_Pax2-GCaMP6f_P8_in vivo_injectable anaesthetic/1_002/processedMovies/1-jumpCorrected.ome.zarr
004  READY    /Volumes/Data/Users/Francesca/Data/2024/2024_07_05_Pax2-GCaMP6f_P8_in vivo_injectable anaesthetic/1_003/processedMovies/1-jumpCorrected.ome.zarr
005  READY    /Volumes/Data/Users/Francesca/Data/2024/2024_07_05_Pax2-GCaMP6f_P8_in vivo_injectable anaesthetic/2/processedMovies/1-jumpCorrected.ome.zarr
006  READY    /Volumes/Data/Users/Francesca/Data/2024/2024_07_05_Pax2-GCaMP6f_P8_in vivo_injectable anaesthetic/2_001/processedMovies/1-jumpCorrected.ome.zarr
007  READY    /Volumes/Data/Users/Francesca/Data/2024/

## 4. Run a dry run, then delete when finished

First run the next cell with `DRY_RUN = True`. It reports the number of stores that would be removed.

When all work using these movies has finished:
1. Review the preview in step 3.
2. Change `DRY_RUN` to `False` and run this cell again.
3. Type the exact confirmation shown at the prompt.

Deletion is permanent. Stop any jobs reading or writing these stores first. The notebook rechecks the paths before deleting and stops on the first error. An interrupted or failed deletion may leave a partly removed store.

A timestamped CSV log is saved locally in `cleanup_logs/`, with an entry before and after each deletion attempt.


In [8]:
DRY_RUN = False  # Change to False only when you are ready to delete.

if preview_settings is None:
    raise RuntimeError("Run the preview in step 3 first.")
if preview_settings != (FILE_LIST, NAS_ROOT, ZARR_NAME):
    raise RuntimeError("Settings changed. Run the preview again.")
if FILE_LIST.read_text() != preview_list_text:
    raise RuntimeError("The file list changed. Run the preview again.")
if not NAS_ROOT.is_dir():
    raise FileNotFoundError("The NAS folder is unavailable. Mount it and preview again.")

if DRY_RUN:
    print(f"DRY RUN: would delete {len(preview_paths)} Zarr stores.")
    print("Nothing was deleted. Review the paths in step 3.")
elif not preview_paths:
    print("No stores are ready for deletion.")
else:
    # Check every candidate before removing the first one.
    for path in preview_paths:
        status, reason = check_target(path, NAS_ROOT)
        if status != "ready":
            raise RuntimeError(f"Preview is no longer valid: {path}\n{reason}\nRun step 3 again.")

    phrase = f"DELETE {len(preview_paths)} ZARR STORES"
    answer = input(
        "Confirm that all work using these movies has finished.\n"
        f"To permanently delete them, type: {phrase}\n"
    )
    if answer != phrase:
        print("Confirmation did not match. Nothing was deleted.")
    else:
        LOG_DIR.mkdir(parents=True, exist_ok=True)
        stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
        log_path = LOG_DIR / f"zarr_cleanup_{stamp}.csv"
        deleted_count = 0

        with log_path.open("x", newline="") as log_file:
            writer = csv.writer(log_file)
            writer.writerow(["time_utc", "path", "status", "detail"])
            log_file.flush()

            def record(path, status, detail=""):
                writer.writerow([
                    datetime.now(timezone.utc).isoformat(), str(path), status, detail
                ])
                log_file.flush()

            print(f"Log: {log_path}")
            for path in preview_paths:
                # Recheck just before deletion as well.
                status, reason = check_target(path, NAS_ROOT)
                if status != "ready":
                    record(path, "stopped", reason)
                    raise RuntimeError(f"Stopped before deleting {path}: {reason}")

                record(path, "started")
                try:
                    shutil.rmtree(path)
                except BaseException as error:
                    record(path, "error_or_interruption", repr(error))
                    print("Stopped. This store may be partly deleted. Check the log.")
                    raise
                record(path, "deleted")
                deleted_count += 1
                print(f"Deleted {deleted_count}/{len(preview_paths)}: {path}")

        print(f"Finished: deleted {deleted_count} Zarr stores.")
        print(f"Deletion log: {log_path}")


Log: ../cleanup_logs/zarr_cleanup_20261111T164607_085855Z.csv
Deleted 1/184: /Volumes/Data/Users/Francesca/Data/2024/2024_07_05_Pax2-GCaMP6f_P8_in vivo_injectable anaesthetic/1/processedMovies/1-jumpCorrected.ome.zarr
Deleted 2/184: /Volumes/Data/Users/Francesca/Data/2024/2024_07_05_Pax2-GCaMP6f_P8_in vivo_injectable anaesthetic/1_001/processedMovies/1-jumpCorrected.ome.zarr
Deleted 3/184: /Volumes/Data/Users/Francesca/Data/2024/2024_07_05_Pax2-GCaMP6f_P8_in vivo_injectable anaesthetic/1_002/processedMovies/1-jumpCorrected.ome.zarr
Deleted 4/184: /Volumes/Data/Users/Francesca/Data/2024/2024_07_05_Pax2-GCaMP6f_P8_in vivo_injectable anaesthetic/1_003/processedMovies/1-jumpCorrected.ome.zarr
Deleted 5/184: /Volumes/Data/Users/Francesca/Data/2024/2024_07_05_Pax2-GCaMP6f_P8_in vivo_injectable anaesthetic/2/processedMovies/1-jumpCorrected.ome.zarr
Deleted 6/184: /Volumes/Data/Users/Francesca/Data/2024/2024_07_05_Pax2-GCaMP6f_P8_in vivo_injectable anaesthetic/2_001/processedMovies/1-jumpCorre

## 5. Check the result

Run this last cell after deletion. It checks the stores that were marked READY in your preview. After a dry run, they will still be present.

Keep the CSV log as a record of what happened. The original path list is left in place. If you rerun the preview after successful cleanup, removed stores will appear as MISSING.


In [9]:
if preview_settings is None:
    raise RuntimeError("Run the preview in step 3 first.")
if not NAS_ROOT.is_dir():
    raise FileNotFoundError("The NAS folder is unavailable; reconnect before checking.")

remaining = [path for path in preview_paths if path.exists() or path.is_symlink()]
print(f"Stores in the preview: {len(preview_paths)}")
print(f"Still present: {len(remaining)}")
for path in remaining:
    print(path)

if preview_paths and not remaining:
    print("All stores marked READY in this preview have been removed.")


Stores in the preview: 184
Still present: 0
All stores marked READY in this preview have been removed.
